In [2]:
import pandas as pd
import plotly.express as px
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
fig1 = px.scatter(
    df, 
    x='age', 
    y='fare', 
    color='sex',  
    hover_data=['pclass', 'alive'],  
    labels={'age': 'Age (Years)', 'fare': 'Fare Paid (£)', 'sex': 'Gender', 'pclass': 'Class', 'alive': 'Survived?'},
    title='Interactive Analysis: Passenger Fare vs. Age'
)

fig1.update_layout(template='plotly_white')

fig1.write_html("titanic_scatter_fare_age.html")
fig1.show()

In [4]:
fig2 = px.histogram(
    df, 
    x='age', 
    color='alive',  
    barmode='overlay',
    hover_data=['pclass', 'sex'],  
    labels={'age': 'Age (Years)', 'count': 'Passenger Count', 'alive': 'Survived?', 'pclass': 'Class', 'sex': 'Gender'},
    title='Interactive Distribution of Passenger Ages'
)

fig2.update_layout(template='plotly_white', yaxis_title='Passenger Count')

fig2.write_html("titanic_histogram_age.html")
fig2.show()

In [5]:
df_grouped = df.groupby(['pclass', 'embark_town'], observed=False).agg(
    average_fare=('fare', 'mean'),
    passenger_count=('fare', 'count')
).reset_index()

fig3 = px.bar(
    df_grouped, 
    x='pclass', 
    y='average_fare', 
    color='embark_town',  
    barmode='group',
    hover_data=['passenger_count', 'embark_town'], 
    labels={'pclass': 'Passenger Class', 'average_fare': 'Average Fare (£)', 'embark_town': 'Embarkation Port', 'passenger_count': 'Total Passengers'},
    title='Interactive Average Ticket Fare by Passenger Class & Port'
)

fig3.update_layout(template='plotly_white')

fig3.write_html("titanic_bar_fare_class.html")
fig3.show()

In [7]:
df_cleaned = df.dropna(subset=['class', 'sex', 'alive'])

fig4 = px.sunburst(
    df_cleaned,
    path=['class', 'sex', 'alive'], 
    color='alive',  
    color_discrete_map={'yes': '#2ca02c', 'no': '#d62728'}, 
    title='Hierarchical Demographics: Survival Breakdown by Class & Gender'
)

fig4.update_layout(template='plotly_white')

fig4.write_html("titanic_sunburst_demographics.html")
fig4.show()

Design Documentation: Advanced Sunburst Chart

1.Why the Sunburst Chart was Chosen
The Titanic dataset contains deeply nested categorical hierarchies (Ticket Class -> Gender -> Survival Outcome) that traditional 2D plots struggle to convey simultaneously. A Sunburst Chart was selected because it naturally visualizes hierarchical, part-to-whole relationships across multiple categorical tracks in a single, space-efficient circular layout.

2.Structural Hierarchy & Path Selection
Outer to Inner Rings: The path parameter was designed as ['class', 'sex', 'alive'].
Placing class at the root immediately divides the dataset by socioeconomic standing.
The subsequent ring breaks each class down by sex, cleanly exposing how the "women and children first" maritime protocol was applied differently across various ticket tiers.

3.Semantic Color Mapping
Instead of standard automated color palettes, a strict categorical map was bound to the alive status: Green (#2ca02c) for survival ('yes') and Red (#d62728) for casualties ('no').
This intentional design use of color theory allows an observer to visually audit the macro-story within two seconds: the third class ring flashes overwhelmingly red, while the first class ring remains predominantly green, particularly for female segments.

4.Interactive Advantages Over Static Frameworks
Unlike static subplots, users can click on any slice (e.g., clicking the "First Class" arc) to completely zoom the visual frame into that specific subsection. The percentages and counts dynamically scale to reflect the active slice, stripping away background noise and providing an on-demand deep dive into subset metrics.